# **Laboratorio 5: Bolsa de Palabras** ⚙️🖼️

**Asignatura:** Visión por Ordenador I  
**Grado:** Ingeniería Matemática e Inteligencia Artificial  
**Curso:** 2026/2027  

### **Objetivo**
En esta parte de la práctica, aprenderá a implementar la Bolsa de Palabras.

## **Librerías**

In [ ]:
! pip install tqdm scikit-learn pandas openpyxl

Solamente si lo anterior no funciona ejecutar esta opción

In [ ]:
import sys
!{sys.executable} -m pip install tqdm scikit-learn pandas openpyxl

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from utils import *
from bow import BoW
from dataset import Dataset
from image_classifier import ImageClassifier
import time
from tqdm import tqdm
import sys
import pickle
import os

# **Apartado D**: Bolsa de palabras

### **Tarea PREVIA:** Generamos la carpeta "../results/partD", junto con la subcarpeta "bow".

In [ ]:
results_path = 
bow_path = 
os.makedirs()

### **Tarea D.1**: Carga de los datasets de entrenamiento y validación para la bolsa de palabras

Descargue el dataset desde el [`enlace en Moodle`](https://sifo.comillas.edu/pluginfile.php/5039369/mod_page/content/8/dataset.zip), descomprimalo en data/partD. 

Empleando el método `load()` de la clase [`Dataset`](.\dataset.py) que se le proporciona, cargue los datasets de entrenamiento y validación.

In [ ]:
# TODO: Load the training and validation sets
training_set = Dataset.load()
validation_set = Dataset.load()

### **Tarea D.2**: Extracción de los descriptores

Para poder crear la bolsa de palabras se va a comenzar obteniendo los descriptores que serán agrupados en las distintas palabras de nuestra bolsa. Para ello complete la carga de la imagen en la ruta `path` en escala de grises y obtenga sus descriptores con los métodos apropiados de cv2. Los descriptores obtenidos se añadirán a nuestra bolsa de palabras (`words`) para después ser agrupados por palabras que representen

In [ ]:
feature_extractor = cv2.SIFT_create()

# Extract features
print("\nComputing SIFT descriptors...")
time.sleep(0.1)  # Prevents a race condition between tqdm and print statements.
descriptors = []
for path in tqdm(training_set, unit="image", file=sys.stdout):
    # TODO: Load the image from the path in grayscale
    image = cv2.imread()
    try:
        # TODO: Using the cv2 methods get the descriptors for the image
        _, descriptor = None
    except:
        print(f"WARN: Issue generating descriptor for image {path}")

    if descriptor is not None:
        descriptors.append(descriptor)

### **Tarea D.3**: Creación del vocabulario

Con los descriptores extraídos se va a realizar su agrupamiento por palabras (K-Means) para obtener un vocabulario de palabras visuales. Agrupar los ~1.5 millones de descriptores del dataset es lento y consume mucha memoria, así que se toma una muestra aleatoria de 70 descriptores por imagen (unos 200 000 en total), que produce un vocabulario igual de útil. Añada la muestra a la bolsa (`words`) con su método `add()`. El vocabulario generado se guarda en un archivo para usarlo en las tareas siguientes.

In [ ]:
# Build vocabulary
vocabulary_size = 100
iterations = 20
termination_criteria = (cv2.TERM_CRITERIA_MAX_ITER | cv2.TERM_CRITERIA_EPS, iterations, 1e-6)
words = cv2.BOWKMeansTrainer(vocabulary_size, termination_criteria)

# Random sample of (at most) 70 descriptors per image (~200 000 in total) to speed up K-means
rng = np.random.default_rng(0)
sample = np.vstack([d[rng.choice(len(d), min(len(d), 70), replace=False)] for d in descriptors])

# TODO: Add the sampled descriptors
words.add()

time.sleep(0.1)  # Prevents a race condition between tqdm and print statements.
print("\nClustering descriptors into", vocabulary_size, "words using K-means...")
vocabulary = words.cluster()
filename=  f"{bow_path}/vocabulary"
with open(filename + ".pickle", "wb") as f:
    pickle.dump(["SIFT", vocabulary], f, pickle.HIGHEST_PROTOCOL)

### **Tarea D.4**: Entrenamiento del clasificador

Una vez se tiene el vocabulario de palabras visuales, se va a entrenar al classificador. Para ello haga uso de la clase [BoW](./bow.py) y complete los métodos que se le indican

In [ ]:
bow = BoW()
# TODO: Especify the args for the loading method
bow.load_vocabulary()

image_classifier = ImageClassifier(bow)
# TODO: Especify the args for the training method
image_classifier.train()
classifier = f"{bow_path}/classifier"
image_classifier.save(classifier)

### **Tarea D.5**: Inferencia en dataset de entrenamiento
Con el modelo entrenado se procede a comprobar su desempeño con el dataset de entrenamiento. Complete los métodos indicados

In [ ]:
bow = BoW()
# TODO: Especify the args for the loading method
bow.load_vocabulary()

image_classifier = ImageClassifier(bow)
# TODO: Especify the args for the loading method
image_classifier.load()
# TODO: Especify the args for the loading method
dataset_name = f"{bow_path}/training"
image_classifier.predict(training_set, dataset_name=dataset_name)

### **Tarea D.6**: Inferencia en dataset de evaluación

A continuación se evalua el desempeño del modelo con el dataset de validación. Complete los métodos indicados

In [ ]:
bow = BoW()
# TODO: Especify the args for the loading method
bow.load_vocabulary()

image_classifier = ImageClassifier(bow)
# TODO: Especify the args for the loading method
image_classifier.load()
# TODO: Especify the args for the loading method
dataset_name = f"{bow_path}/validation"
image_classifier.predict(validation_set, dataset_name=dataset_name)

### **Pregunta D.1**: Búsqueda de los mejores parámetros. ¿Cuántas palabras usar?

Entrene y evalúe el clasificador con vocabularios de 50, 100, 200 y 400 palabras (`vocabulary_size`). Elija además **uno** de estos parámetros y estudie su efecto: el kernel del SVM (`setKernel()` en `image_classifier.py`), el número de iteraciones del SVM (argumento `iterations` de `train()`) o el extractor de características (KAZE en lugar de SIFT). Muestre en una tabla la *accuracy* de entrenamiento y de validación de cada prueba y justifique qué configuración elegiría.

In [ ]:
#TODO: Homework